In [1]:
import os
import subprocess

In [2]:
LANG_SRC = "python"
LANG_DST = "go"

MODEL_TYPE = "roberta"
PRETRAINED_MODEL = "microsoft/unixcoder-base"

N_EPOCHS = 15
LEARNING_RATE = 5e-5

BEAM_SIZE = 10
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 128

BATCH_SIZE_EVAL = 256
BATCH_SIZE_TRAIN = 256

MINI_DATASET_SIZE = 4096
MINI_MODE_ENABLED = False

FILENAME_JSONL_TEST = "test.jsonl"
FILENAME_JSONL_TRAIN = "train.jsonl"
FILENAME_JSONL_VALID = "valid.jsonl"

FILENAME_JSONL_TEST_MINI = "test-mini.jsonl"
FILENAME_JSONL_TRAIN_MINI = "train-mini.jsonl"
FILENAME_JSONL_VALID_MINI = "valid-mini.jsonl"

FILENAME_CSV_BLEU_SCORES = "bleu_scores.csv"
FILENAME_CSV_EVAL_LOSSES = "eval_losses.csv"
FILENAME_CSV_TRAIN_LOSSES = "train_losses.csv"
FILENAME_TXT_BLEU_SCORE_TEST = "bleu_score.test"

In [3]:
ARCHIVE_NOTEBOOK_DIRNAME = "codexglue-summarization-cross-python-on-go"
_archive_search_root = os.getcwd()
notebook_path = None
while True:
    _candidate = os.path.join(
        _archive_search_root
        , "notebooks"
        , "archive"
        , "codexglue_summarization"
        , ARCHIVE_NOTEBOOK_DIRNAME
    )
    if os.path.isdir(_candidate):
        notebook_path = os.path.abspath(_candidate)
        break
    _parent = os.path.dirname(_archive_search_root)
    if _parent == _archive_search_root:
        break
    _archive_search_root = _parent

if notebook_path is None:
    raise RuntimeError("Could not locate archived notebook folder: {name}".format(name=ARCHIVE_NOTEBOOK_DIRNAME))
root_path = os.path.dirname(notebook_path)

repo_path = os.path.join(root_path, "repos", "CodeXGLUE")
task_path = os.path.join(repo_path, "Code-Text", "code-to-text")

src_root_path = os.path.join(root_path, "src", "python")
src_task_path = os.path.join(src_root_path, "codexglue", "summarization")

code_path = os.path.join(task_path, "code")
dataset_path = os.path.join(task_path, "dataset")
evaluator_path = os.path.join(task_path, "evaluator")

model_path = os.path.join(notebook_path, "model")
src_notebook_path = os.path.join(notebook_path, "src")

model_name = "{dataset}-{task}-{model_type}-{pretrained_model}-{lang}-n_epochs={n_epochs}-lr={lr}" \
    .format(
        lang=LANG_SRC
        , lr=LEARNING_RATE
        , n_epochs=N_EPOCHS
        , dataset="codexglue"
        , task="summarization"
        , model_type=MODEL_TYPE
        , pretrained_model=PRETRAINED_MODEL.replace("/", "_").replace("-", "_")
    )

def _archive_display_path(path):
    try:
        return os.path.relpath(path, start=root_path)
    except ValueError:
        return path



In [4]:
ARCHIVE_RERUN_ENABLED = False
ARCHIVE_SUBPROCESS_TIMEOUT_SECONDS = 6 * 60 * 60
checkpoint_path = os.path.join(model_path, "checkpoint-best-bleu", "pytorch_model.bin")
run_py_path = os.path.join(code_path, "run.py")
test_filename_path = os.path.join(
    dataset_path
    , LANG_DST
    , FILENAME_JSONL_TEST if not MINI_MODE_ENABLED else FILENAME_JSONL_TEST_MINI
)

if not ARCHIVE_RERUN_ENABLED:
    print("Archive rerun disabled; skipping CodeXGLUE test evaluation.")
elif not os.path.exists(run_py_path):
    print("Archived CodeXGLUE checkout not found at {path}; skipping test evaluation.".format(path=_archive_display_path(run_py_path)))
    print("Restore the original CodeXGLUE repo at {path} to rerun this archived evaluation.".format(path=_archive_display_path(repo_path)))
elif not os.path.exists(checkpoint_path):
    print("Archived checkpoint not found at {path}; skipping test evaluation.".format(path=_archive_display_path(checkpoint_path)))
    print("Run the training cells or restore the historical model/ directory before comparing outputs.")
elif not os.path.exists(test_filename_path):
    print("Archived test dataset not found at {path}; skipping test evaluation.".format(path=_archive_display_path(test_filename_path)))
else:
    subprocess.check_call(
        [
            "python"
            
            , run_py_path
            
            , "--model_type", MODEL_TYPE
            , "--model_name_or_path", PRETRAINED_MODEL
            , "--output_dir", model_path
            , "--load_model_path", checkpoint_path

            , "--beam_size", str(BEAM_SIZE)
            , "--max_source_length", str(MAX_SOURCE_LEN)
            , "--max_target_length", str(MAX_TARGET_LEN)

            , "--do_test"
            , "--eval_batch_size", str(BATCH_SIZE_EVAL)
            , "--test_filename", test_filename_path

            , "--bleu_score_test_txt_filename", os.path.join(model_path, FILENAME_TXT_BLEU_SCORE_TEST)
        ]
        , timeout=ARCHIVE_SUBPROCESS_TIMEOUT_SECONDS
    )


Archive rerun disabled; skipping CodeXGLUE test evaluation.


In [8]:
bleu_score_path = os.path.join(model_path, FILENAME_TXT_BLEU_SCORE_TEST)

if not os.path.exists(bleu_score_path):
    print("Archived BLEU score file not found at {path}; skipping score preview.".format(path=_archive_display_path(bleu_score_path)))
else:
    with open(bleu_score_path, 'r') as bleu_score_test_file:
        print("Test Bleu score: ", bleu_score_test_file.readline())


Archived BLEU score file not found at codexglue-summarization-cross-python-on-go/model/bleu_score.test; skipping score preview.


In [9]:
print("Top 10 test data ground truth summaries of code in {lang_dst} language:\n".format(lang_dst=LANG_DST))

_gold_path = os.path.join(model_path, "test_0.gold")
if not os.path.exists(_gold_path):
    print("Archived ground-truth summary file not found at {path}; skipping preview.".format(path=_archive_display_path(_gold_path)))
else:
    with open(_gold_path, mode="r") as f:
        for _ in range(10):
            print(f.readline())


Top 10 test data ground truth summaries of code in go language:

Archived ground-truth summary file not found at codexglue-summarization-cross-python-on-go/model/test_0.gold; skipping preview.


In [10]:
print("Top 10 test data predicted summaries of code in {lang_dst} language using the trained {model_name} model:\n".format(lang_dst=LANG_DST, model_name=model_name))

_output_path = os.path.join(model_path, "test_0.output")
if not os.path.exists(_output_path):
    print("Archived prediction file not found at {path}; skipping preview.".format(path=_archive_display_path(_output_path)))
else:
    with open(_output_path, mode="r") as f:
        for _ in range(10):
            print(f.readline())


Top 10 test data predicted summaries of code in go language using the trained codexglue-summarization-roberta-microsoft_unixcoder_base-python-n_epochs=15-lr=5e-05 model:

Archived prediction file not found at codexglue-summarization-cross-python-on-go/model/test_0.output; skipping preview.
